# Adversarial Robustness in Deep Learning for Image Classification

This notebook builds a research-style PyTorch project in Google Colab for CIFAR-10 classification with attack generation, defense through adversarial training, model comparison, robustness analysis, rich visualizations, and an interactive Gradio demo.

## What This Notebook Demonstrates

- Fast demo-friendly Custom CNN training on CIFAR-10
- FGSM and PGD adversarial attacks
- 10k image subset training for quick turnaround in Colab
- Robustness curves and grouped comparison charts
- Screenshot-ready image panels, perturbation heatmaps, and misclassification examples
- Automatic printed analysis for presentation or viva discussion
- Gradio UI with image upload, model toggle, attack toggle, epsilon slider, and prediction confidence

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/Aashi-ghub/AdversarialDefenseNet.git'
WORKSPACE_ROOT = Path('/content')
REPO_ROOT = WORKSPACE_ROOT / 'AdversarialDefenseNet'
PROJECT_ROOT = REPO_ROOT / 'adv_project'

if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Repository root: {REPO_ROOT}')
print(f'Project root: {PROJECT_ROOT}')


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gradio>=4.0'], check=True)
print('Installed Gradio for the interactive demo section.')


In [ ]:
import json
from IPython.display import Markdown, display

import matplotlib.pyplot as plt
import pandas as pd
import torch

from evaluation.analysis import (
    build_defense_comparison_table,
    build_model_comparison_table,
    build_results_dataframe,
    generate_analysis_report,
    save_analysis_report,
)
from evaluation.robustness import accuracy_vs_epsilon, build_attack_fn, robustness_suite
from evaluation.visualization import (
    plot_attack_comparison_panel,
    plot_clean_vs_adversarial_bar_chart,
    plot_defense_comparison_chart,
    plot_model_comparison_graph,
    plot_misclassification_examples,
    plot_perturbation_heatmaps,
    plot_robustness_curves,
    plot_training_curves,
)
from models.cnn import CustomCNN
from models.resnet import build_resnet18
from training.adversarial_training import PGDAdversarialTrainer
from training.dataloader import get_cifar10_dataloaders
from training.trainer import Trainer, build_criterion, build_optimizer_and_scheduler
from ui.gradio_app import build_gradio_interface
from utils.config import build_config, get_device
from utils.logger import MetricLogger, setup_logger

FAST_DEMO = True
STANDARD_EPOCHS = 2 if FAST_DEMO else 4
ADVERSARIAL_EPOCHS = 1 if FAST_DEMO else 3

config = build_config(PROJECT_ROOT)
config.experiment.train_epochs = STANDARD_EPOCHS
config.experiment.adv_train_epochs = ADVERSARIAL_EPOCHS
config.experiment.run_models = ('cnn',)
config.experiment.adv_models = ()
config.data.batch_size = 256
config.data.num_workers = 2
config.data.train_subset_size = 10000
config.data.val_subset_size = 2000
config.attack.fgsm_curve_epsilons = [0.0, 2 / 255, 4 / 255, 8 / 255, 12 / 255, 16 / 255]
config.attack.pgd_steps = 7 if FAST_DEMO else 10
config.attack.pgd_alpha = 2 / 255
config.save(PROJECT_ROOT / 'outputs' / 'run_config.json')

device = get_device(config)
print(f'Using device: {device}')
print(json.dumps(config.to_dict(), indent=2))


In [ ]:
train_loader, val_loader, test_loader, class_names = get_cifar10_dataloaders(config)

display(Markdown('### Dataset Summary'))
print(f'Train batches: {len(train_loader)}')
print(f'Validation batches: {len(val_loader)}')
print(f'Test batches: {len(test_loader)}')
print(f'Classes: {class_names}')


In [ ]:
def create_model(model_name: str):
    if model_name == 'cnn':
        return CustomCNN(num_classes=config.experiment.num_classes)
    if model_name == 'resnet18':
        return build_resnet18(num_classes=config.experiment.num_classes, use_pretrained=False)
    raise ValueError(f'Unsupported model: {model_name}')


def create_standard_trainer(model, experiment_name: str):
    optimizer, scheduler = build_optimizer_and_scheduler(model, config, config.experiment.train_epochs)
    logger = setup_logger('trainer', PROJECT_ROOT / 'outputs' / 'logs', experiment_name)
    metric_logger = MetricLogger(PROJECT_ROOT / 'outputs' / 'metrics' / f'{experiment_name}.csv')
    return Trainer(
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        criterion=build_criterion(),
        device=device,
        checkpoint_dir=PROJECT_ROOT / 'checkpoints',
        logger=logger,
        metric_logger=metric_logger,
        use_amp=config.experiment.use_amp,
    )


def create_adversarial_trainer(model, experiment_name: str):
    optimizer, scheduler = build_optimizer_and_scheduler(model, config, config.experiment.adv_train_epochs)
    logger = setup_logger('adv_trainer', PROJECT_ROOT / 'outputs' / 'logs', experiment_name)
    metric_logger = MetricLogger(PROJECT_ROOT / 'outputs' / 'metrics' / f'{experiment_name}.csv')
    return PGDAdversarialTrainer(
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        criterion=build_criterion(),
        device=device,
        checkpoint_dir=PROJECT_ROOT / 'checkpoints',
        mean=config.data.mean,
        std=config.data.std,
        epsilon=config.attack.pgd_epsilon,
        alpha=config.attack.pgd_alpha,
        steps=config.attack.pgd_steps,
        logger=logger,
        metric_logger=metric_logger,
        use_amp=config.experiment.use_amp,
    )


## Fast Training: CNN Baseline

In [ ]:
trained_models = {}
histories = {}

for model_name in config.experiment.run_models:
    experiment_name = f'{model_name}_standard'
    print(f'Training {experiment_name} ...')
    model = create_model(model_name).to(device)
    trainer = create_standard_trainer(model, experiment_name)
    history = trainer.fit(train_loader, val_loader, config.experiment.train_epochs, experiment_name)
    trained_models[experiment_name] = model
    histories[experiment_name] = history


In [ ]:
cnn_checkpoint_path = PROJECT_ROOT / 'checkpoints' / 'cnn_standard_best.pth'
cnn_checkpoint = torch.load(cnn_checkpoint_path, map_location='cpu')
cnn_state_dict = cnn_checkpoint['model_state_dict'] if 'model_state_dict' in cnn_checkpoint else cnn_checkpoint
torch.save(cnn_state_dict, REPO_ROOT / 'model.pth')
print(f'Exported CNN weights for the web app to: {REPO_ROOT / "model.pth"}')


## Optional Adversarial Defense

In [ ]:
if config.experiment.adv_models:
    for model_name in config.experiment.adv_models:
        experiment_name = f'{model_name}_adv_pgd'
        print(f'Adversarial training {experiment_name} ...')
        model = create_model(model_name).to(device)
        trainer = create_adversarial_trainer(model, experiment_name)
        history = trainer.fit(train_loader, val_loader, config.experiment.adv_train_epochs, experiment_name)
        trained_models[experiment_name] = model
        histories[experiment_name] = history
else:
    print('Skipping adversarial training to keep the live demo fast.')


## Robustness Evaluation

In [ ]:
robustness_results = {}
for experiment_name, model in trained_models.items():
    robustness_results[experiment_name] = robustness_suite(model, test_loader, device, config)

results_df = build_results_dataframe(robustness_results)
results_df.to_csv(PROJECT_ROOT / 'outputs' / 'metrics' / 'research_results.csv', index=False)

model_table = build_model_comparison_table(results_df)
defense_table = build_defense_comparison_table(results_df)

display(Markdown('### Core Model Comparison'))
display(model_table)
display(Markdown('### CNN Defense Comparison'))
display(defense_table)
display(Markdown('### Full Experiment Metrics'))
display(results_df[[
    'display_name',
    'clean_accuracy_pct',
    'fgsm_accuracy_pct',
    'pgd_accuracy_pct',
    'fgsm_drop_pct',
    'pgd_drop_pct',
]])


## Training Curves and Comparison Charts

In [ ]:
training_curve_path = PROJECT_ROOT / 'outputs' / 'plots' / 'training_curves.png'
model_comparison_path = PROJECT_ROOT / 'outputs' / 'plots' / 'cnn_vs_resnet_comparison.png'
clean_vs_adv_path = PROJECT_ROOT / 'outputs' / 'plots' / 'clean_vs_adversarial_accuracy.png'
defense_chart_path = PROJECT_ROOT / 'outputs' / 'plots' / 'cnn_defense_comparison.png'

plot_training_curves(histories, training_curve_path, title='Training Curves: Standard vs Adversarial Training')
plot_model_comparison_graph(results_df, model_comparison_path, title='CNN vs ResNet-18 Robustness Comparison')
plot_clean_vs_adversarial_bar_chart(results_df, clean_vs_adv_path, title='Clean vs Adversarial Accuracy Across Experiments')
plot_defense_comparison_chart(results_df, defense_chart_path, title='Defense Effect: Standard CNN vs PGD-Trained CNN')
plt.show()

print(f'Saved training curves to: {training_curve_path}')
print(f'Saved model comparison chart to: {model_comparison_path}')
print(f'Saved clean-vs-adversarial chart to: {clean_vs_adv_path}')
print(f'Saved defense chart to: {defense_chart_path}')


## Accuracy vs Epsilon Curves

In [ ]:
fgsm_curves = {}
pgd_curves = {}

for experiment_name, model in trained_models.items():
    fgsm_curves[experiment_name] = accuracy_vs_epsilon(
        model=model,
        data_loader=test_loader,
        device=device,
        attack_name='fgsm',
        epsilons=config.attack.fgsm_curve_epsilons,
        mean=config.data.mean,
        std=config.data.std,
    )
    pgd_curves[experiment_name] = accuracy_vs_epsilon(
        model=model,
        data_loader=test_loader,
        device=device,
        attack_name='pgd',
        epsilons=config.attack.fgsm_curve_epsilons,
        mean=config.data.mean,
        std=config.data.std,
        alpha=config.attack.pgd_alpha,
        steps=config.attack.pgd_steps,
    )

fgsm_curve_path = PROJECT_ROOT / 'outputs' / 'plots' / 'fgsm_accuracy_vs_epsilon.png'
pgd_curve_path = PROJECT_ROOT / 'outputs' / 'plots' / 'pgd_accuracy_vs_epsilon.png'

plot_robustness_curves(fgsm_curves, fgsm_curve_path, title='FGSM Accuracy vs Epsilon')
plot_robustness_curves(pgd_curves, pgd_curve_path, title='PGD Accuracy vs Epsilon')
plt.show()

print(f'Saved FGSM epsilon curve to: {fgsm_curve_path}')
print(f'Saved PGD epsilon curve to: {pgd_curve_path}')


## Advanced Visualizations for Screenshots

In [ ]:
reference_experiment = 'cnn_adv_pgd' if 'cnn_adv_pgd' in trained_models else 'cnn_standard'
reference_model = trained_models[reference_experiment]

fgsm_attack_fn = build_attack_fn(
    attack_name='fgsm',
    epsilon=config.attack.fgsm_epsilon,
    mean=config.data.mean,
    std=config.data.std,
)
pgd_attack_fn = build_attack_fn(
    attack_name='pgd',
    epsilon=config.attack.pgd_epsilon,
    alpha=config.attack.pgd_alpha,
    steps=config.attack.pgd_steps,
    mean=config.data.mean,
    std=config.data.std,
)

comparison_panel_path = PROJECT_ROOT / 'outputs' / 'plots' / f'{reference_experiment}_comparison_panel.png'
heatmap_panel_path = PROJECT_ROOT / 'outputs' / 'plots' / f'{reference_experiment}_heatmaps.png'
misclassification_panel_path = PROJECT_ROOT / 'outputs' / 'plots' / f'{reference_experiment}_misclassifications.png'

plot_attack_comparison_panel(
    model=reference_model,
    data_loader=test_loader,
    device=device,
    fgsm_fn=fgsm_attack_fn,
    pgd_fn=pgd_attack_fn,
    class_names=class_names,
    mean=config.data.mean,
    std=config.data.std,
    output_path=comparison_panel_path,
    num_images=4,
)
plot_perturbation_heatmaps(
    model=reference_model,
    data_loader=test_loader,
    device=device,
    fgsm_fn=fgsm_attack_fn,
    pgd_fn=pgd_attack_fn,
    mean=config.data.mean,
    std=config.data.std,
    output_path=heatmap_panel_path,
    num_images=4,
)
plot_misclassification_examples(
    model=reference_model,
    data_loader=test_loader,
    device=device,
    fgsm_fn=fgsm_attack_fn,
    pgd_fn=pgd_attack_fn,
    class_names=class_names,
    mean=config.data.mean,
    std=config.data.std,
    output_path=misclassification_panel_path,
    num_images=4,
)
plt.show()

print(f'Saved image comparison panel to: {comparison_panel_path}')
print(f'Saved perturbation heatmaps to: {heatmap_panel_path}')
print(f'Saved misclassification panel to: {misclassification_panel_path}')


## Automatic Analysis Section

In [ ]:
analysis_report = generate_analysis_report(results_df)
analysis_path = PROJECT_ROOT / 'outputs' / 'reports' / 'analysis_report.txt'
save_analysis_report(analysis_report, analysis_path)

display(Markdown('### Research Insights'))
print(analysis_report)
print(f'\nSaved report to: {analysis_path}')


## Interactive Gradio Demo

In [ ]:
import gradio as gr

demo_models = {}
if 'cnn_standard' in trained_models:
    demo_models['CNN'] = trained_models['cnn_standard']
if 'resnet18_standard' in trained_models:
    demo_models['ResNet-18'] = trained_models['resnet18_standard']
if 'cnn_adv_pgd' in trained_models:
    demo_models['Adv-CNN (PGD Defense)'] = trained_models['cnn_adv_pgd']

demo = build_gradio_interface(
    model_registry=demo_models,
    class_names=class_names,
    config=config,
    device=device,
)
demo


In [ ]:
demo.launch(share=True)


## Saved Artifacts

In [ ]:
checkpoint_files = sorted(str(path.relative_to(PROJECT_ROOT)) for path in (PROJECT_ROOT / 'checkpoints').glob('*.pth'))
output_files = sorted(
    str(path.relative_to(PROJECT_ROOT))
    for path in (PROJECT_ROOT / 'outputs').rglob('*')
    if path.is_file()
)

print('Saved checkpoints:')
for file_name in checkpoint_files:
    print(f'  - {file_name}')

print('\nSaved outputs:')
for file_name in output_files:
    print(f'  - {file_name}')
